In [ ]:
#0 Montar o Drive e definir a pasta do dataset (a MESMA usada no notebook 02)
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/multimodais/dados'
os.makedirs(DATA_DIR, exist_ok=True)
print('DATA_DIR:', DATA_DIR)
# Pré-requisito: o arquivo `fontes.csv` (do repo) precisa estar nesta pasta do Drive.
assert os.path.exists(os.path.join(DATA_DIR, 'fontes.csv')), f'Suba fontes.csv para {DATA_DIR}'

In [ ]:
#1 Baixar as imagens do fontes.csv para DATA_DIR (pula o que já existe)
# Commons bloqueia (429) download em massa dos ORIGINAIS -> baixamos a MINIATURA,
# resolvendo a thumburl exata via API (1 chamada, sem rate-limit). Met/Flickr: link direto.
import csv, json, re, time, urllib.request, urllib.parse, urllib.error
UA = {'User-Agent': 'AtelieGenerativo/1.0 (projeto academico UNICEUB; diego.morais@mundodosdados.com.br)'}

csv_path = os.path.join(DATA_DIR, 'fontes.csv')
rows = [r for r in csv.DictReader(open(csv_path, encoding='utf-8')) if r.get('arquivo')]

def commons_title(u):
    m = re.search(r'/wiki/(File:.+)$', u or '')
    return urllib.parse.unquote(m.group(1)).replace('_', ' ') if m else None

titles = list({commons_title(r['url']) for r in rows if commons_title(r.get('url', ''))})
thumb_by_title = {}
for i in range(0, len(titles), 45):
    api = 'https://commons.wikimedia.org/w/api.php?' + urllib.parse.urlencode({
        'action': 'query', 'format': 'json', 'prop': 'imageinfo',
        'iiprop': 'url', 'iiurlwidth': '1280', 'titles': '|'.join(titles[i:i+45])})
    d = json.load(urllib.request.urlopen(urllib.request.Request(api, headers=UA), timeout=60))
    for p in d['query']['pages'].values():
        if 'imageinfo' in p:
            thumb_by_title[p['title'].replace('_', ' ')] = p['imageinfo'][0].get('thumburl')

def baixar(url, dest, tries=4):
    for k in range(tries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=90) as resp, open(dest, 'wb') as out:
                out.write(resp.read())
            return
        except urllib.error.HTTPError as e:
            if e.code == 429 and k < tries - 1:
                wait = int(e.headers.get('Retry-After', 0)) or 5 * (k + 1) ** 2
                print(f'      429 -> aguardando {wait}s'); time.sleep(wait); continue
            raise

for i, r in enumerate(rows, 1):
    dest = os.path.join(DATA_DIR, r['arquivo'])
    if os.path.exists(dest):
        continue
    t = commons_title(r.get('url', ''))
    url = thumb_by_title.get(t) if t else r.get('url_imagem', '')
    if not url or not url.startswith('http'):
        print(f'[{i:02d}/{len(rows)}] sem URL utilizável: {r["arquivo"]}'); continue
    try:
        baixar(url, dest)
        print(f'[{i:02d}/{len(rows)}] ok  {r["arquivo"]}')
    except Exception as e:
        print(f'[{i:02d}/{len(rows)}] ERRO {r["arquivo"]}: {e}')
    time.sleep(0.5)

imgs = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(('.jpg', '.png'))]
print(f'\nImagens em DATA_DIR: {len(imgs)}')
assert imgs, f'Nenhuma imagem baixada em {DATA_DIR}.'

In [ ]:
#2 Legendas (captions) em INGLÊS com BLIP + token do estilo -> metadata.jsonl
# BLIP gera em inglês (ideal p/ SD-1.5). Prefixamos com o token de gatilho do estilo.
# IMPORTANTE: estes são RASCUNHOS. A revisão humana é obrigatória (Etapa 1 da sistematização):
# reveja o legendas.txt, corrija descrições erradas e rode a célula #3 para regravar o metadata.
!pip -q install transformers pillow
import csv, json, os
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

TOKEN = 'estilo_lambelambe'   # palavra-gatilho do estilo (a mesma no treino e no app)

proc = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
blip = BlipForConditionalGeneration.from_pretrained(
    'Salesforce/blip-image-captioning-base').to('cuda')

rows = [r for r in csv.DictReader(open(os.path.join(DATA_DIR, 'fontes.csv'), encoding='utf-8')) if r.get('arquivo')]
meta_path = os.path.join(DATA_DIR, 'metadata.jsonl')
leg_path = os.path.join(DATA_DIR, 'legendas.txt')

n = 0
with open(meta_path, 'w', encoding='utf-8') as fj, open(leg_path, 'w', encoding='utf-8') as fl:
    fl.write('# arquivo<TAB>caption  (rascunho BLIP em inglês; REVISAR manualmente e re-rodar a célula #3)\n')
    for r in rows:
        fn = r['arquivo']
        p = os.path.join(DATA_DIR, fn)
        if not os.path.exists(p):
            continue
        img = Image.open(p).convert('RGB')
        out = blip.generate(**proc(img, return_tensors='pt').to('cuda'), max_new_tokens=40)
        cap = proc.decode(out[0], skip_special_tokens=True).strip()
        caption = f'{TOKEN}, {cap}'                       # ex.: "estilo_lambelambe, an old photo of a woman in a dress"
        fj.write(json.dumps({'file_name': fn, 'text': caption}, ensure_ascii=False) + '\n')
        fl.write(f'{fn}\t{caption}\n')
        print(f'{fn} -> {caption}')
        n += 1

print(f'\n{n} legendas geradas. metadata.jsonl e legendas.txt salvos em {DATA_DIR}')
print('>>> Agora REVISE o legendas.txt à mão e rode a célula #3 para regravar o metadata.jsonl <<<')

In [ ]:
#3 (Após revisar o legendas.txt à mão) regrava o metadata.jsonl a partir dele
# Fluxo: rode #2 (rascunho BLIP) -> edite o legendas.txt corrigindo as descrições -> rode esta célula.
import json, os

leg_path = os.path.join(DATA_DIR, 'legendas.txt')
meta_path = os.path.join(DATA_DIR, 'metadata.jsonl')

n = 0
with open(meta_path, 'w', encoding='utf-8') as fj:
    for line in open(leg_path, encoding='utf-8'):
        line = line.rstrip('\n')
        if not line or line.startswith('#') or '\t' not in line:
            continue
        fn, caption = line.split('\t', 1)
        fn, caption = fn.strip(), caption.strip()
        if not os.path.exists(os.path.join(DATA_DIR, fn)):
            print('aviso: imagem inexistente, pulando ->', fn); continue
        fj.write(json.dumps({'file_name': fn, 'text': caption}, ensure_ascii=False) + '\n')
        n += 1

# Verificação final do pareamento imagens <-> metadata
meta = [json.loads(l)['file_name'] for l in open(meta_path, encoding='utf-8')]
imgs = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(('.jpg', '.png'))]
faltando = [m for m in meta if not os.path.exists(os.path.join(DATA_DIR, m))]
print(f'metadata.jsonl regravado: {n} linhas | imagens: {len(imgs)} | faltando: {len(faltando)}')
print('Dataset pronto. Pode ir para o notebook 02 (treino).')